In [4]:
import numpy as np

from articles_dataset import ArticlesDataset

articles_dataset_path = \
    "../resources/bcc-news-entertainment.csv"
dataset = ArticlesDataset(articles_dataset_path)
dataset.load_articles(50)
dataset.preprocess()

sents_count = 0
for article in dataset.articles:
    sents_count += len(list(article.document.sents))
print("Total sentences count acrros loaded documents: ", sents_count)


Total sentences count acrros loaded documents:  743


In [2]:
from transformers import pipeline

pipe = pipeline(model="facebook/bart-large-mnli")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'I have a problem with my iphone that needs to be resolved asap!',
 'labels': ['urgent', 'phone', 'computer', 'not urgent', 'tablet'],
 'scores': [0.522758424282074,
  0.458139568567276,
  0.014264951460063457,
  0.002685005310922861,
  0.002152078552171588]}

In [6]:
candidates_labels = ["film", "television", "music", "theatre", "books", "celebrity", "awards", "art"]

In [14]:
import pandas as pd
import numpy as np

rows = []
for article in dataset.articles:
    for sentence in article.document.sents:
        result = pipe(sentence.text, candidate_labels=candidates_labels)
        labels = result["labels"]
        scores = result["scores"]
        highest = np.argmax(result["scores"])
        rows.append({
            "sentence": sentence.text,
            "label": labels[highest],
            "score": scores[highest],
        })

zero_shot_classification_df = pd.DataFrame(rows)

In [15]:
zero_shot_classification_df

,sentence,label,score
0,French musician Jean-Michel Jarre is to perfo...,music,0.762312
1,Denmark is holding a three-day celebration of ...,music,0.432823
2,Other stars are expected to join the line-up i...,celebrity,0.893057
3,"""Christian Andersen's fairy tales are timeless...",art,0.641907
4,"""For all of us, at any age there is always - b...",books,0.205095
...,...,...,...
738,If I'm half the person you are and half the ta...,celebrity,0.319810
739,Both Foxx and Swank are now considered to be a...,awards,0.410394
740,"However, Swank has to overcome a strong challe...",theatre,0.533885
741,"Meanwhile, ballots for Oscars - the Hollywood'...",awards,0.516368


In [16]:
zero_shot_classification_df.to_csv("../results/zero_shot_classification.csv", index=False)